# XGBoost Hyperparameter Tuning

This notebook extracts the XGBoost search that produced the configuration retained by the deployed California home-price model. It excludes unfinished deployment cells and alternative-model experiments from the source notebooks.

The archived searches ran on an earlier 18-feature list-unaware frame. The final application uses a reduced 12-feature contract, so the archived selections are model-development evidence rather than a claim that the complete search was rerun after feature reduction. The selected configuration is re-fitted and evaluated on the final processed frames below.

In [1]:
import os
from math import prod

import numpy as np
import pandas as pd
from sklearn.metrics import make_scorer, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

FEATURES = [
    'DaysOnMarket',
    'Latitude',
    'Longitude',
    'BathroomsTotalInteger',
    'LivingArea',
    'FireplaceYN',
    'YearBuilt',
    'ParkingTotal',
    'BedroomsTotal',
    'PoolPrivateYN',
    'LotSizeAcres',
    'Stories',
]

XGB_DEVICE = os.environ.get('XGB_DEVICE', 'cpu')
RUN_FULL_TUNING = os.environ.get('RUN_FULL_TUNING', '0') == '1'
SEARCH_JOBS = 1 if XGB_DEVICE == 'cuda' else -1

print(f'XGBoost device: {XGB_DEVICE}')
print(f'Full tuning enabled: {RUN_FULL_TUNING}')

XGBoost device: cpu
Full tuning enabled: False


## Final processed data contract

`trn.csv` contains the January-August training frame and `tst.csv` contains the September-October temporal test frame. These processed listing-level files are excluded from the repository. Only the 12 deployed features and the sale-price target are used below; listing-price fields remain available solely for the separate benchmark in `modeling.ipynb`.

In [2]:
trn = pd.read_csv('trn.csv')
tst = pd.read_csv('tst.csv')

required_columns = FEATURES + ['ClosePrice', 'OriginalListPrice', 'ListPrice']
for name, frame in [('training', trn), ('test', tst)]:
    missing = sorted(set(required_columns) - set(frame.columns))
    if missing:
        raise ValueError(f'{name} frame is missing required columns: {missing}')
    if frame[FEATURES + ['ClosePrice']].isna().any().any():
        raise ValueError(f'{name} frame contains missing model values')
    if (frame['ClosePrice'] <= 0).any():
        raise ValueError(f'{name} frame contains non-positive sale prices')

for frame in (trn, tst):
    frame[['FireplaceYN', 'PoolPrivateYN']] = frame[
        ['FireplaceYN', 'PoolPrivateYN']
    ].astype(int)

X_trn = trn[FEATURES].copy()
X_tst = tst[FEATURES].copy()
y_trn = trn['ClosePrice'].to_numpy(dtype=float)
y_tst = tst['ClosePrice'].to_numpy(dtype=float)
y_trn_log = np.log(y_trn)

print(f'Training shape: {X_trn.shape}')
print(f'Test shape: {X_tst.shape}')
print(f'Feature order: {list(X_trn.columns)}')

Training shape: (78162, 12)
Test shape: (21246, 12)
Feature order: ['DaysOnMarket', 'Latitude', 'Longitude', 'BathroomsTotalInteger', 'LivingArea', 'FireplaceYN', 'YearBuilt', 'ParkingTotal', 'BedroomsTotal', 'PoolPrivateYN', 'LotSizeAcres', 'Stories']


## Search objectives

The source tuning code compared mean and median absolute percentage error on the natural-log target used for training. These are selection criteria in log space, not dollar-space MAPE and MdAPE. Dollar-space metrics are calculated only after applying the exponential inverse transform.

The cleaned code retains the archived objectives so the documented parameter selections remain interpretable.

In [3]:
def median_log_target_ape(y_true_log, y_pred_log):
    y_true_log = np.asarray(y_true_log, dtype=float)
    y_pred_log = np.asarray(y_pred_log, dtype=float)
    return float(np.median(np.abs((y_true_log - y_pred_log) / y_true_log)))


mean_log_ape_scorer = make_scorer(
    mean_absolute_percentage_error,
    greater_is_better=False,
)
median_log_ape_scorer = make_scorer(
    median_log_target_ape,
    greater_is_better=False,
)

## Stage 1: broad search

The broad grid varies tree depth, learning rate, and estimator count while holding row and column sampling at 0.8. It contains 125 candidate combinations and requires 625 fits per scoring criterion with five-fold cross-validation.

In [4]:
broad_grid = {
    'max_depth': [3, 5, 7, 9, 11],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'n_estimators': [100, 300, 500, 800, 1000],
}

assert prod(len(values) for values in broad_grid.values()) == 125

## Stage 2: refined search

The refined grid concentrates on the plateau identified by the broad searches. It contains 48 combinations and requires 240 fits per scoring criterion.

In [5]:
refined_grid = {
    'max_depth': [7, 9, 11, 13],
    'learning_rate': [0.01, 0.05, 0.10],
    'n_estimators': [1000, 1100, 1200, 1300],
}

assert prod(len(values) for values in refined_grid.values()) == 48

## Archived search selections

The source notebooks retain the following completed `GridSearchCV` outputs:

| Stage | Selection criterion | Depth | Learning rate | Estimators |
| --- | --- | ---: | ---: | ---: |
| Broad | Mean log-target percentage error | 7 | 0.05 | 1,000 |
| Broad | Median log-target percentage error | 11 | 0.01 | 1,000 |
| Refined | Mean log-target percentage error | 7 | 0.05 | 1,300 |
| Refined | Median log-target percentage error | 11 | 0.01 | 1,300 |

The refined mean-error result is the configuration retained by the deployed model. Because the archived searches used the earlier expanded feature frame, this table is kept separate from the final 12-feature temporal test metrics.

## Optional exhaustive execution

The four searches total 1,730 model fits. They are disabled during normal notebook execution so routine verification does not repeat a long model-selection run. Set `RUN_FULL_TUNING=1` before starting the kernel to execute them. On a CUDA system, set `XGB_DEVICE=cuda`; search-level parallelism is then limited to one job to avoid competing GPU fits.

In [6]:
def make_search(param_grid, scorer):
    estimator = XGBRegressor(
        objective='reg:squarederror',
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=1,
        device=XGB_DEVICE,
    )
    return GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scorer,
        cv=5,
        n_jobs=SEARCH_JOBS,
        verbose=2,
    )


search_plan = {
    'broad_mean_log_ape': (broad_grid, mean_log_ape_scorer),
    'broad_median_log_ape': (broad_grid, median_log_ape_scorer),
    'refined_mean_log_ape': (refined_grid, mean_log_ape_scorer),
    'refined_median_log_ape': (refined_grid, median_log_ape_scorer),
}

search_results = {}
if RUN_FULL_TUNING:
    for name, (grid, scorer) in search_plan.items():
        search = make_search(grid, scorer)
        search.fit(X_trn, y_trn_log)
        search_results[name] = {
            'best_params': search.best_params_,
            'best_score': -float(search.best_score_),
        }
    print(pd.DataFrame(search_results).T.to_string())
else:
    print('Full 1,730-fit search skipped. Set RUN_FULL_TUNING=1 to execute it.')

Full 1,730-fit search skipped. Set RUN_FULL_TUNING=1 to execute it.


## Selected configuration on the final feature contract

This bounded step re-fits only the selected configuration on the final 12-feature training frame and reports dollar-space performance on the September-October temporal test frame. It verifies code compatibility without repeating model selection.

In [7]:
selected_params = {
    'objective': 'reg:squarederror',
    'max_depth': 7,
    'learning_rate': 0.05,
    'n_estimators': 1300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1,
    'device': XGB_DEVICE,
}

selected_model = XGBRegressor(**selected_params)
selected_model.fit(X_trn, y_trn_log)
y_pred = np.exp(selected_model.predict(X_tst))

evaluation = pd.DataFrame(
    {
        'Metric': ['R²', 'MAPE', 'MdAPE'],
        'Temporal-test result': [
            r2_score(y_tst, y_pred),
            mean_absolute_percentage_error(y_tst, y_pred) * 100,
            np.median(np.abs((y_tst - y_pred) / y_tst)) * 100,
        ],
    }
)
evaluation['Temporal-test result'] = evaluation['Temporal-test result'].round(2)
evaluation

,Metric,Temporal-test result
0,R²,0.90
1,MAPE,11.19
2,MdAPE,7.75


## Interpretation boundary

The selected-configuration result above validates the final feature order, training call, inverse transform, and aggregate metric calculations. It does not represent a new exhaustive search. The deployed UBJSON artifact remains the authoritative runtime model and is independently protected by checksum, feature-order, and deterministic-prediction tests.